# Baseline: NER on Clean CoNLL-2003

Evaluate three tokenizers on clean CoNLL-2003 NER data.  
This is **Phase 1** of the case study — the reference point before adding noise.

**Tokenizers:**
- `bert-base-uncased` — WordPiece
- `gpt2` — Byte Pair Encoding (BPE)
- `google/byt5-small` — Byte-level (no vocabulary, no OOV)

**Metrics:**
- Tokenizer: fertility (tokens/word), OOV rate, vocab coverage
- NER: seqeval entity-level F1 (standard CoNLL metric)


## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

from metrics import compute_tokenizer_stats, print_stats_table
from train import get_model_and_tokenizer, tokenize_and_align_labels, make_compute_metrics

# ── Device ─────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

# ── Config ─────────────────────────────────────────────────────────────────────
MODEL_NAMES = [
    "bert-base-uncased",
    "gpt2",
    "google/byt5-small",
]

RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "results", "tables")
os.makedirs(RESULTS_DIR, exist_ok=True)


Device: mps


## 1. Load CoNLL-2003

Standard English NER benchmark. Labels: `O`, `B/I-PER`, `B/I-ORG`, `B/I-LOC`, `B/I-MISC`


In [2]:
dataset = load_dataset("conll2003", trust_remote_code=True)

label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)

print(f"Labels ({num_labels}): {label_names}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")


Labels (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train: 14041 | Val: 3250 | Test: 3453


In [3]:
# Preview a few examples
for ex in dataset["train"].select(range(3)):
    tokens = ex["tokens"]
    tags = [label_names[t] for t in ex["ner_tags"]]
    print(" | ".join(f"{tok}/{tag}" for tok, tag in zip(tokens, tags)))
    print()


EU/B-ORG | rejects/O | German/B-MISC | call/O | to/O | boycott/O | British/B-MISC | lamb/O | ./O

Peter/B-PER | Blackburn/I-PER

BRUSSELS/B-LOC | 1996-08-22/O



## 2. Tokenizer Analysis

Measure how each tokenizer handles clean CoNLL-2003 text.

- **Fertility** — avg tokens produced per word (higher = more fragmentation)
- **OOV rate** — fraction of words producing at least one UNK token
- **UNK rate** — fraction of all tokens that are UNK


In [4]:
from transformers import AutoTokenizer

sentences = [ex["tokens"] for ex in dataset["train"]]

tokenizer_stats = {}
for model_name in MODEL_NAMES:
    print(f"Analyzing {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    stats = compute_tokenizer_stats(tok, sentences, sample_size=3000)
    tokenizer_stats[model_name] = stats

print()
print_stats_table(tokenizer_stats)


Analyzing bert-base-uncased ...
Analyzing gpt2 ...
Analyzing google/byt5-small ...

Tokenizer              Vocab Size  Fertility   OOV Rate   UNK Rate
------------------------------------------------------------------
bert-base-uncased           30522      1.267     0.0000     0.0000
gpt2                        50257      1.304     0.0000     0.0000
google/byt5-small             256      4.417     0.0000     0.0000


In [5]:
# Save tokenizer stats to CSV
stats_df = pd.DataFrame(tokenizer_stats).T.reset_index()
stats_df.columns = ["model", "fertility", "oov_rate", "unk_rate", "vocab_size"]
stats_df.to_csv(os.path.join(RESULTS_DIR, "tokenizer_stats_clean.csv"), index=False)
print("Saved tokenizer_stats_clean.csv")
stats_df


Saved tokenizer_stats_clean.csv


,model,fertility,oov_rate,unk_rate,vocab_size
0,bert-base-uncased,1.267,0.0,0.0,30522.0
1,gpt2,1.304,0.0,0.0,50257.0
2,google/byt5-small,4.417,0.0,0.0,256.0


## 3. NER Fine-tuning

Train each model on CoNLL-2003 train set, evaluate on test set.

> **Note:** Training all three models takes ~30–90 min on MPS depending on hardware.
> You can reduce `num_train_epochs` for quick iteration.


### 3.1 BERT (WordPiece)

In [6]:
MODEL_NAME = "bert-base-uncased"
model, tokenizer = get_model_and_tokenizer(MODEL_NAME, num_labels, id2label, label2id)

tokenized_bert = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenized. Example token count:", len(tokenized_bert["train"][0]["input_ids"]))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

Tokenized. Example token count: 512


In [7]:
training_args = TrainingArguments(
    output_dir=f"../results/checkpoints/{MODEL_NAME.replace('/', '_')}",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer_bert = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_bert["train"],
    eval_dataset=tokenized_bert["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=make_compute_metrics(label_names),
)

trainer_bert.train()


/Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.070456,0.052325,0.919036,0.910759,0.927465


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=878, training_loss=0.1338328299598433, metrics={'train_runtime': 1454.1555, 'train_samples_per_second': 9.656, 'train_steps_per_second': 0.604, 'total_flos': 3669099951393792.0, 'train_loss': 0.1338328299598433, 'epoch': 1.0})

In [11]:
trainer_bert.callback_handler.on_train_begin(
    trainer_bert.args, trainer_bert.state, trainer_bert.control
)
trainer_bert.eval_dataset = tokenized_bert["test"]
bert_results = trainer_bert.evaluate()
print(f"BERT  |  F1: {bert_results['eval_f1']:.4f}  |  "
      f"Precision: {bert_results['eval_precision']:.4f}  |  "
      f"Recall: {bert_results['eval_recall']:.4f}")


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.070456,0.092892,0.890836,0.881720,0.900142


BERT  |  F1: 0.8908  |  Precision: 0.8817  |  Recall: 0.9001


### 3.2 GPT-2 (BPE)

In [12]:
MODEL_NAME = "gpt2"
model_gpt2, tokenizer_gpt2 = get_model_and_tokenizer(MODEL_NAME, num_labels, id2label, label2id)

tokenized_gpt2 = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer_gpt2),
    batched=True,
    remove_columns=dataset["train"].column_names,
)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForTokenClassification LOAD REPORT from: gpt2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [13]:
training_args_gpt2 = TrainingArguments(
    output_dir=f"../results/checkpoints/gpt2",
    num_train_epochs=1,
    per_device_train_batch_size=8,   # GPT-2 uses more memory
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer_gpt2 = Trainer(
    model=model_gpt2,
    args=training_args_gpt2,
    train_dataset=tokenized_gpt2["train"],
    eval_dataset=tokenized_gpt2["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer_gpt2),
    compute_metrics=make_compute_metrics(label_names),
)

trainer_gpt2.train()


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.159411,0.160748,0.711260,0.691745,0.731908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.25453120972409604, metrics={'train_runtime': 1657.0675, 'train_samples_per_second': 8.473, 'train_steps_per_second': 1.06, 'total_flos': 3669099951393792.0, 'train_loss': 0.25453120972409604, 'epoch': 1.0})

In [15]:
trainer_gpt2.callback_handler.on_train_begin(
    trainer_gpt2.args, trainer_gpt2.state, trainer_gpt2.control
)
gpt2_results = trainer_gpt2.evaluate(tokenized_gpt2["test"])
print(f"GPT-2 |  F1: {gpt2_results['eval_f1']:.4f}  |  "
      f"Precision: {gpt2_results['eval_precision']:.4f}  |  "
      f"Recall: {gpt2_results['eval_recall']:.4f}")


/Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.159411,0.196603,0.687350,0.666113,0.709986


GPT-2 |  F1: 0.6874  |  Precision: 0.6661  |  Recall: 0.7100


### 3.3 ByT5 (Byte-level)

ByT5 tokenizes text into individual bytes — no vocabulary needed, zero OOV by definition.
We use the T5 encoder + linear classification head (`ByT5ForTokenClassification`).

> ByT5 produces ~3-4x more tokens per sentence than WordPiece, so we reduce batch size.


In [21]:
def tokenize_and_align_labels_byt5(examples, tokenizer, max_length=512):
    all_input_ids, all_attention_masks, all_labels = [], [], []
    for tokens, ner_tags in zip(examples["tokens"], examples["ner_tags"]):
        input_ids = [0]  # <bos> у byt5 нет, но pad/eos есть
        label_ids = [-100]
        for word, label in zip(tokens, ner_tags):
            word_ids = tokenizer(word, add_special_tokens=False)["input_ids"]
            input_ids += word_ids
            label_ids += [label] + [-100] * (len(word_ids) - 1)
        input_ids = input_ids[:max_length]
        label_ids = label_ids[:max_length]
        pad_len = max_length - len(input_ids)
        all_input_ids.append(input_ids + [0] * pad_len)
        all_attention_masks.append([1]*len(input_ids) + [0]*pad_len)
        all_labels.append(label_ids + [-100]*pad_len)
    return {"input_ids": all_input_ids, "attention_mask": all_attention_masks, "labels": all_labels}

In [24]:
MODEL_NAME = "google/byt5-small"
model_byt5, tokenizer_byt5 = get_model_and_tokenizer(MODEL_NAME, num_labels, id2label, label2id)

tokenized_byt5 = dataset.map(
    lambda ex: tokenize_and_align_labels_byt5(ex, tokenizer_byt5, max_length=256),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenized. Example token count:", len(tokenized_byt5["train"][0]["input_ids"]))


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

T5EncoderModel LOAD REPORT from: google/byt5-small
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenized. Example token count: 256


In [25]:
training_args_byt5 = TrainingArguments(
    output_dir=f"../results/checkpoints/byt5",
    num_train_epochs=1,
    per_device_train_batch_size=1,   # byte-level = long sequences
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer_byt5 = Trainer(
    model=model_byt5,
    args=training_args_byt5,
    train_dataset=tokenized_byt5["train"],
    eval_dataset=tokenized_byt5["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer_byt5, pad_to_multiple_of=8),
    compute_metrics=make_compute_metrics(label_names),
)

trainer_byt5.train()


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.349028,0.065053,0.907736,0.901397,0.914165


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.encoder.embed_tokens.weight'].


TrainOutput(global_step=3511, training_loss=0.7528769064750878, metrics={'train_runtime': 1897.2171, 'train_samples_per_second': 7.401, 'train_steps_per_second': 1.851, 'total_flos': 4682304557336064.0, 'train_loss': 0.7528769064750878, 'epoch': 1.0})

In [26]:
trainer_byt5.callback_handler.on_train_begin(
    trainer_byt5.args, trainer_byt5.state, trainer_byt5.control
)
byt5_results = trainer_byt5.evaluate(tokenized_byt5["test"])
print(f"ByT5  |  F1: {byt5_results['eval_f1']:.4f}  |  "
      f"Precision: {byt5_results['eval_precision']:.4f}  |  "
      f"Recall: {byt5_results['eval_recall']:.4f}")


/Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.349028,0.208363,0.862590,0.853591,0.871781


ByT5  |  F1: 0.8626  |  Precision: 0.8536  |  Recall: 0.8718


## 4. Baseline Results Summary

Reference F1 scores on **clean** CoNLL-2003 test set.
This table will be extended in notebook `03_noisy_evaluation.ipynb`.


In [27]:
summary = pd.DataFrame([
    {
        "model": "bert-base-uncased",
        "strategy": "WordPiece",
        **tokenizer_stats["bert-base-uncased"],
        "f1_clean": bert_results["eval_f1"],
        "precision_clean": bert_results["eval_precision"],
        "recall_clean": bert_results["eval_recall"],
    },
    {
        "model": "gpt2",
        "strategy": "BPE",
        **tokenizer_stats["gpt2"],
        "f1_clean": gpt2_results["eval_f1"],
        "precision_clean": gpt2_results["eval_precision"],
        "recall_clean": gpt2_results["eval_recall"],
    },
    {
        "model": "google/byt5-small",
        "strategy": "Byte-level",
        **tokenizer_stats["google/byt5-small"],
        "f1_clean": byt5_results["eval_f1"],
        "precision_clean": byt5_results["eval_precision"],
        "recall_clean": byt5_results["eval_recall"],
    },
])

summary.to_csv(os.path.join(RESULTS_DIR, "baseline_results.csv"), index=False)
print("Saved baseline_results.csv")
summary


Saved baseline_results.csv


,model,strategy,fertility,oov_rate,unk_rate,vocab_size,f1_clean,precision_clean,recall_clean
0,bert-base-uncased,WordPiece,1.267,0.0,0.0,30522,0.890836,0.881720,0.900142
1,gpt2,BPE,1.304,0.0,0.0,50257,0.687350,0.666113,0.709986
2,google/byt5-small,Byte-level,4.417,0.0,0.0,256,0.862590,0.853591,0.871781
